# **Infer a mask of the Endolysosomal System From Endolysosomal Organelles**

***Prior to this notebook, you should have already run through [1.0_image_setup](1.0_image_setup.ipynb) and any segmentations from workflows 1.2-1.7 that you wish to include when creating the combined endolysosomal mask.***

### ➡️ **Input:**
In this workflow, ".tiff" segmentations files of multiple different organelles will be combined into a single mask. This idea can be extended to any combination of organelles, but the use case that we will discuss here is the creation of a combined endo-lysosomal mask that encompasses organelles in the endo-lysosomal system, such as lysosomes and endosomes (which can be segmented using workflows suited for round organelles, such as lipid droplets, lysosomes, and peroxisomes). 

### **Output:** ➡️
The output from this workflow will be a single-channel ".tiff" file containing the segmentation of individual endolysosomes compartments (each compartment has a unique ID number). You can save the segmentation file at the end of the notebook for a single image if desired, or proceed to the [batch_process_segmentation](/infer-subc/notebooks/part_1_segmentation_workflows/batch_process_segmentations.ipynb) notebook to apply the settings determined below to a batch of images. One single-channel ".tiff" file will be exported per input image in the batch processing step.

### 🍃 **Biological relevance - Endolysosomal compartment**
The endolysosomal system is made up of several functionally unique organelles, including lysosomes and a variety of endosomes, that coordinate to fulfill different trafficking within a cell. These compartments are unique compared to other organelles in that they combine and mix regularly to facilitate transport of materials.

We are created a combined endolysosomal system mask to better understand the composition and functionality of the endolysosomal compartments. The combined mask will add together the volume segmented in each of the endolysosome compartments. Then, the mask can be used to determine which fluorescently tagged markers (from the raw intensity image) and organelle objects (from the segmentations) are mixed within the same endolysosome compartment and to what degree. 

-----

### 👣 **Summary of steps**

➡️ **EXTRACTION**
- **`STEP 1`** - Select the organelles to combine

**PRE-PROCESSING**
- **`STEP 2`** - Create binary masks from each segmentation image

**CORE PROCESSING**
- **`STEP 3`** - Combine the organelle masks into one

**POST-PROCESSING**
- **`STEP 4`** - Declumping the endolysosomal mask

**POST-POST-PROCESSING** - None

**EXPORT** ➡️
- Save stacked masks to output file location


---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`1.0_image_setup`](1.0_image_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from typing import List, Union
import time

from infer_subc.core.file_io import (list_image_files, 
                                     read_czi_image, 
                                     read_tiff_image,
                                     export_inferred_organelle,
                                     sample_input)
from infer_subc.utils.batch import find_segmentation_tiff_files
from infer_subc.core.img import select_cellmask_from_img
from infer_subc.organelles.cellmask import (find_radius, 
                                            infer_soma_from_mask, 
                                            infer_neurites_from_mask, 
                                            clean_soma_from_neurites, 
                                            clean_neurites_from_soma)

from skimage.measure._label import label
import skimage
from skimage.feature import peak_local_max
import scipy.ndimage as ndi

import napari
from napari.utils.notebook_display import nbscreenshot
import seaborn as sns

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: `sample_data_type`, `data_root_path`, `in_data_path`, `im_type`, and `out_data_path`.

In [ ]:
### USER INPUT REQUIRED ###
sample_data_type = None


# Edit locations indicated by "USER SPECIFIED".
data_root_path = Path(r"Y:\Images for Shannon_From Kajal")

in_data_path = data_root_path / "raw"

im_type = ".tiff"

out_data_path = data_root_path / "seg"

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# If sample_data_type is set to "neuron_1", "astrocyte", "neuron_2" or "iPSC" then the sample data is used and the directories are set
if sample_data_type != None:
    data_root_path, im_type, in_data_path, out_data_path = sample_input(sample_data_type)

# Create the output directory to save the segmentation outputs in.
if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
img_file_list = list_image_files(in_data_path,im_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":img_file_list})

#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: `test_img_n`

In [ ]:
#### USER INPUT REQUIRED ###
test_img_n = 0

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# Read in the image and metadata as an ndarray and dictionary from the test image selected above. 
test_img_name = img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Define some of the metadata features and print them.
channel_names = meta_dict['name']
meta = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
file_path = meta_dict['file_name']

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale (ZYX): {scale}")
print(f"Channel axis: {channel_axis}")

# open viewer and add images
viewer = napari.Viewer()
for i in list(range(len(channel_names))):
    viewer.add_image(img_data[i],
                     scale=scale,
                     name=f"Channel {i}")
viewer.grid.enabled = True
viewer.reset_view()
print("\nProceed to Napari window to view your selected image.")

# screenshot viewer
nbscreenshot(viewer, canvas_only = True)

-----

## **EXTRACTION**

### **`STEP 1` - Select the organelles to combine**

&#x1F453; **FYI:** In this step, the organelle segmentation .tiff files that will be combined into the endolysosomal mask are selected.

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the organelle names you wish to include:
- `orgs_to_combine`: a list of organelle names (that match the suffixes of the segmentation files) that you wish to combine into a single mask

In [ ]:
#### USER INPUT REQUIRED ###
orgs_to_combine = ['eea1lyso', 'golgi', 'lysolyso', 'rab11lyso']

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block selects the organelle segmentation images that will be used as the input for the downstream analysis and adds it to the Napari viewer.

In [ ]:
# find the paths to the segmentation images in the out_data_path and read them into a dictionary
seg_paths = find_segmentation_tiff_files(file_path, orgs_to_combine, out_data_path, '')
seg_dict = {org: read_tiff_image(seg_paths[org]) for org in orgs_to_combine}


# visualize the cell mask and the raw intensity image in napari
viewer.layers.clear()
viewer.grid.enabled = True
for org_name, org_seg in seg_dict.items():
    viewer.add_labels(org_seg, scale=scale, name=f"{org_name} segmentation image")

print("\nProceed to Napari window to view the segmentation outputs.")

-----
## **PRE-PROCESSING**

### **`STEP 2` - Create binary masks from each segmentation image**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This step will binarize each segmentation to generate a mask of each input organelle.

In [ ]:
# binarize the segmentation images and print the number of objects in each original segmentation image and the new binary mask
binary_seg_dict = {org: (seg > 0).astype(int) for org, seg in seg_dict.items()}

print("The original organelle segmentations containing many individual objects are now binary masks:")
for org_name in orgs_to_combine:
    print(f"{org_name} had {len(np.unique(seg_dict[org_name]))-1} objects --> now {len(np.unique(binary_seg_dict[org_name]))-1}")

# visualize the binary masks in napari
viewer.layers.clear()
viewer.grid.enabled = False
colors = sns.color_palette("tab10")
for org_name, org_seg in binary_seg_dict.items():
    viewer.add_image(org_seg, scale=scale, name=f"{org_name} binary mask", colormap=colors[orgs_to_combine.index(org_name)], blending='additive')

print("\nProceed to Napari window to view the binary organelle masks.")

-----
## **CORE-PROCESSING**

### **`STEP 3` - Combine the organelle masks into one**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** The binary organelle masks are combined into a preliminary endolysosomal mask in the cell below.

In [ ]:
# combine all organelles into one mask
combined_mask = np.zeros_like(list(binary_seg_dict.values())[0])
for org_seg in binary_seg_dict.values():
    combined_mask = np.logical_or(combined_mask, org_seg)

# adding image to Napari as a new layer
viewer.add_image(combined_mask, name="Combined Endolyso Mask", scale=scale, blending='additive', opacity=0.5)
print("The combined mask has been added to the Napari viewer as a new layer (white). It should only appear in any voxels that contain one of the input organelles.")

-----
## **POST-PROCESSING**

### **`STEP 4` - Declumping the endolysosomal mask**

&#x1F453; **FYI:** This code block takes the combined mask and creates an `instance segmentation` separating the individual compartments from each other. In this output, each individual object in the image is given a unique ID number. 

In this workflow objects may be separated based on `connectivity`: if a pixel/voxel is touching another pixel/voxel in any direction, they are considered the same object and each pixel/voxel within that object is labeled as the same unique ID number. Alternatively,  objects may be separated using a morphological `declumping` approach that finds centerpoints within the mask area that represent the further distance from the edge of the mask. Those center points are then used as "seeds" to separate individual objects by [watersheding](https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_watershed.html/).

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `watershed`: A True/False statement of whether to use the watershed declumping method or not. 
    - If True, the morphology-based watershed approach is used.
    - If False, the connectivity method is instead used. 
    - If None, no declumping is performed at all
- `footprint_size`: The size of the footprint used to identify the "local maxima" in the distance transform image. Larger values will likely select fewer seeds

In [ ]:
watershed = True
footprint_sz = 8

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block creates an instance segmentation using the settings provided above.

*In the Napari viewer, the image is added as a "labels" layer where each object appears as a different color.*

In [ ]:
# create instance segmentation based on connectivity
if watershed == None:
    combined_seg = combined_mask.astype(int)
elif watershed == False:
    combined_seg = label(combined_mask)
elif watershed == True:
    # distance transform approach to ID seeds (footprint = 80 is good)
    combined_labels = label(combined_mask)
    distance = ndi.distance_transform_edt(combined_labels, sampling=scale)
    viewer.add_image(distance, name="Intermediate: distance transform image", scale=scale, blending='additive', colormap='magma')

    # hard footprint based on ZYX scale format
    coords = peak_local_max(distance, footprint=np.ones((round((footprint_sz*scale[0])/scale[1]), footprint_sz, footprint_sz)), labels=combined_labels)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers = label(mask)
    viewer.add_image(markers, name="Intermediate: seeds detected from distance transform (based on footprint size)", scale=scale, blending='additive')

    combined_seg = skimage.segmentation.watershed(-distance, markers, mask=combined_labels)

# adding image to Napari as a new layer
viewer.add_labels(combined_seg, scale=scale, name="Combined Endolyso Segmentation")

print("The declumped segmentation is now added to the Napari viewer.")
print("If watershed was chosen, the intermediate steps are also included for easier setting optimization.")

-----
## **POST-POST-PROCESSING**

No post-post-processing steps included

-----
## **SAVING**

## **`Saving` - Save the segmentation output**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block saves the instance segmentation output to the `out_data_path` specified earlier.

In [ ]:
# Saving file
out_file_n = export_inferred_organelle(combined_seg, "endolyso_seg", meta_dict, out_data_path)
print(f"Saved to: {out_data_path}")

-----
-----
## **Define `infer_soma_neurites()` function**
The following code includes an example of how the workflow steps above are combined into one function. This function can be run below to process a single image. It is included in the [batch process notebook](batch_process_segmentations.ipynb) to run the above segmentation on multiple images. 

This function can be utilized from infer-subc using:
```python
infer_subc.organelles.cellmask.infer_soma_neurites()
```

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block defines a prototype of the `infer_soma_neurites()` function. It is applied below.

In [ ]:
def _infer_endolyso_mask(org_seg_list: List[np.ndarray],
                         org_name_list: List[str],
                         watershed: bool,
                         footprint_sz: int) -> np.ndarray:
    """
    This function takes in a list of organelles to combine into an single mask; the combined mask can be declumped in the final step 
    to define individual object.

    Parameters
    ----------
    org_seg_list: List[np.ndarray]
        A list of organelle segmentation masks to combine.
    org_name_list: List[str]
        A list of names corresponding to the organelle segmentation masks.
    watershed: bool
        Whether to apply a watershed algorithm to the combined mask to declump it into individual objects.
        True = apply watershed
        False = create instance segmentation based on connectivity
        None = no declumping, a binary mask will be saved as the output
    footprint_sz: int
        The size of the footprint to use for the distance transform when applying the watershed algorithm. This is only used if 
        watershed=True. The optimal size will depend on the size of the objects being combined and the scale of the image. 
        A larger footprint will result in fewer seeds and therefore larger segmented objects, while a smaller footprint will result 
        in more seeds and therefore smaller segmented objects.
    """

    ###################
    # EXTRACT
    ###################  
    seg_dict = {org: org_seg for org, org_seg in zip(org_name_list, org_seg_list)}

    ###################
    # PRE_PROCESSING
    ################### 
    binary_seg_dict = {org: (seg > 0).astype(int) for org, seg in seg_dict.items()}

    ###################
    # CORE_PROCESSING
    ###################
    combined_mask = np.zeros_like(list(binary_seg_dict.values())[0])
    for org_seg in binary_seg_dict.values():
        combined_mask = np.logical_or(combined_mask, org_seg)

    ###################
    # POST_PROCESSING
    ################### 
    if watershed == None:
        combined_seg = combined_mask.astype(int)
    elif watershed == False:
        combined_seg = label(combined_mask)
    elif watershed == True:
        # distance transform approach to ID seeds (footprint = 80 is good)
        combined_labels = label(combined_mask)
        distance = ndi.distance_transform_edt(combined_labels, sampling=scale)
        viewer.add_image(distance, name="Intermediate: distance transform image", scale=scale, blending='additive', colormap='magma')

        # hard footprint based on ZYX scale format
        coords = peak_local_max(distance, footprint=np.ones((round((footprint_sz*scale[0])/scale[1]), footprint_sz, footprint_sz)), labels=combined_labels)
        mask = np.zeros(distance.shape, dtype=bool)
        mask[tuple(coords.T)] = True
        markers = label(mask)
        viewer.add_image(markers, name="Intermediate: seeds detected from distance transform (based on footprint size)", scale=scale, blending='additive')

        combined_seg = skimage.segmentation.watershed(-distance, markers, mask=combined_mask)

    ###################
    # POST_POST_PROCESSING
    ################### 
    # none

    return combined_seg

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [ ]:
combo_seg = _infer_endolyso_mask(org_seg_list=list(seg_dict.values()),
                                 org_name_list=orgs_to_combine,
                                 watershed=watershed,
                                 footprint_sz=footprint_sz)

#confirms this output matches the output saved above
print(f"The segmentation output here matches the output created above: {np.all(combo_seg == combined_seg)}")

#### &#x1F3C3; **Run code; no user input required** 

<mark>**This is not yet implemented in the main infer-subc code, only in this notebook**

&#x1F453; **FYI:** This code block imports the matching function contained within `infersubc` and tests it using the same settings specified above.

In [ ]:
# # import soma_neurite function from infer_subc.organelles.cellmask and run soma and neurite segmentation function 
# from infer_subc.organelles.endolyso import infer_endolyso_mask

# endolyso_seg = infer_endolyso_mask(org_name_list=orgs_to_combine,
#                                     org_seg_list=list(seg_dict.values()),
#                                     watershed=watershed,
#                                     footprint_sz=footprint_sz)


# #confirm this output matches the output saved above
# print(f"The segmentation output here matches the output created above: {np.all(endolyso_seg == combined_seg)}")

# # adding image to Napari as a new layer
# viewer.layers.clear()
# viewer.add_labels(endolyso_seg, scale=scale, name="Endolysosomal Segmentation from Function")
# viewer.reset_view()

# # screenshot viewer
# nbscreenshot(viewer, canvas_only = False)

------------
### **BATCH PROCESSING**

<mark> **THIS SECTION IS TEMPORARY** - once the workflows and functions have been integrated into infer-subc, this section will be removed and batch processing can be done in the plugin or batch processing notebook.

#### **Define batch function for endolysosomal mask creation**

In [ ]:
# define the batch function
def _batch_process_endolyso_seg(raw_path: Union[Path,str],
                               raw_file_type: str,
                               seg_path: Union[Path, str],
                               name_suffix: Union[str, None],
                               endolyso_settings: Union[List, None]):
    """
    This function batch processes the segmentation workflows for multiple organelles and masks across multiple images.

    Parameters:
    ----------
    raw_path: Union[Path,str]
        A string or a Path object of the path to your raw (e.g., intensity) images that will be the input for segmentation
    raw_file_type: str
        The raw file type (e.g., ".tiff" or ".czi")
    seg_path: Union[Path, str]
        A string or a Path object of the path where the segmentation outputs will be saved 
    name_suffix: str
        An optional string to include before the segmentation suffix at the end of the output file. 
        For example, if the name_suffix was "20240105", the segmentation file output from the 1.1_masks workflow would include:
        "{base-file-name}-20240105-masks"
    endolyso_settings: Union[List, None]
        The necessary settings for the endolysosomal segmentation function in the form of a list.
        Ex: [['golgi, 'lyso', 'perox', 'LD'], True, 20] representing the organelle names, whether to apply watershed, and the 
        footprint size for the distance transform when applying watershed.
        
        NOTE: the organelle segmentations list (the first input to the infer_endolyso_mask function) will be created in the function
        below. Do not include it in this list!

    Returns:
    ----------
    None

    The endolysosomal segmentations are saved in the specified directory.

    """
    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)

    if not Path.exists(seg_path):
        Path.mkdir(seg_path)
        print(f"The specified 'seg_path' was not found. Creating {seg_path}.")
    
    if not name_suffix:
        name_suffix=""

    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    for img in img_file_list:
        count = count + 1
        print(f"Beginning segmentation of: {img}")
        seg_list = []
        mask = None

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(img)

        if endolyso_settings:
            seg_paths = find_segmentation_tiff_files(img, endolyso_settings[0], seg_path, name_suffix)
            seg_list = [read_tiff_image(seg_paths[org]) for org in endolyso_settings[0]]
            endolyso_mask = _infer_endolyso_mask(seg_list, *endolyso_settings)
            export_inferred_organelle(endolyso_mask, name_suffix+"endolyso_seg", meta_dict, seg_path)  
            seg_list.append("endolyso")

        end = time.time()
        print(f"Processing for {img} completed in {(end - start)/60} minutes.")

    return print(f"Batch processing complete: {count} images segmented in {(end-start)/60} minutes.")

#### **Execute batch processing of endolysosomal segmentation**

In [ ]:
### USER INPUT REQUIRED ###
raw_path = in_data_path
raw_file_type = im_type
seg_path = out_data_path
name_suffix = ""
endolyso_settings = [orgs_to_combine, watershed, footprint_sz]

In [ ]:
# execute batch processing function with user inputs
_batch_process_endolyso_seg(raw_path, raw_file_type, seg_path, name_suffix, endolyso_settings)


-------------
### ✅ **INFER ENDOLYSOSOMAL MASK IS COMPLETE!**
